# 05 — Case Studies từ `_evolution_log.txt`

Thực hiện **Section 8**: dùng evolution log để chọn 2–3 case study minh hoạ
(không dùng làm evidence thống kê chính).

> ⚠️ Tương tự notebook 04, `parse_evolution_log` dùng regex placeholder.
> Kiểm thử trên file mẫu thật ở cell 5.1 trước khi chạy hàng loạt.


In [1]:
# ==== CẤU HÌNH ĐƯỜNG DẪN (chỉnh lại cho đúng máy của bạn) ====
import sys, os
sys.path.append(os.path.abspath("."))  # để import evrp_analysis_utils.py cùng thư mục

FULL_DIR  = "benchmark/full"     # thư mục kết quả bản đầy đủ (equity-aware)
NOEQ_DIR  = "benchmark/no-EQ"    # thư mục kết quả bản loại equity guidance
ARTIFACT_DIR = "artifacts"       # nơi lưu các bảng trung gian (csv) giữa các notebook
os.makedirs(ARTIFACT_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import evrp_analysis_utils as utils

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [2]:
manifest = pd.read_csv(f"{ARTIFACT_DIR}/run_manifest_matched_valid.csv")
td_solutions = pd.read_csv(f"{ARTIFACT_DIR}/td_solutions.csv")


## 5.1. Kiểm thử parser trên một file mẫu

In [3]:
sample_run = manifest.iloc[0]
sample_path = os.path.join(
    sample_run["RunDir"], f"{sample_run['Instance']}_seed_{int(sample_run['Seed'])}_evolution_log.txt"
)
if os.path.exists(sample_path):
    with open(sample_path, encoding="utf-8", errors="ignore") as f:
        preview = "".join([next(f) for _ in range(20)])
    print(preview)
    ev = utils.parse_evolution_log(sample_path)
    print(f"\n{len(ev)} sự kiện parse được")
    ev.head()
else:
    print(f"Không tìm thấy {sample_path}")


--- ALNS Solution Evolution Log ---
(Logs all accepted non-dominated/dominating solutions)

----------------------------------------
[Iteration: 100 | Result: Rejected]
[Operators: TimeSlackDestroy / Greedy Energy Insertion (Enhanced)]
----------------------------------------
Solution (Vehicles: 13, Distance: 1240.17, WorkloadGini: 0.14, MaxTime: 1199.38, Feasible: true)
--- Route ID: 4 --- Feasible: YES
   Total Distance:       55.13
   Total Time:          900.62
   Total Energy Cons:    55.13
--- Node Sequence (Count: 11) ---
  Idx | StrID |       Type |  ArrTime |  DepTime |   RemBat |  RemLoad |   Charge
---------------------------------------------------------------------------------------
    0 |    D0 |      Depot |     0.00 |     0.00 |    79.69 |   200.00 |     0.00
   62 |   C41 |   Customer |    18.68 |   139.00 |    61.01 |   190.00 |     0.00
   61 |   C40 |   Customer |   141.00 |   233.00 |    59.01 |   180.00 |     0.00
   63 |   C42 |   Customer |   235.83 |   329.00 

## 5.2. Tiêu chí chọn case study (theo Section 8 của tài liệu)

1. Cùng fleet, distance gần nhau giữa FULL/No-EQ nhưng Gini giảm mạnh ở FULL.
2. Instance mà equity-focused relocation ban đầu làm battery/TW khó hơn (nhìn qua evolution trace).
3. Instance cho thấy FULL hội tụ dần đến workload cân bằng hơn theo thời gian.

In [4]:
pivot = td_solutions.pivot_table(index="Instance", columns="Variant", values=[c for c in ["Z2", "Z3"] if c in td_solutions.columns])

if ("Z2", "FULL") in pivot.columns and ("Z2", "No-EQ") in pivot.columns:
    pivot["Distance_gap_pct"] = (pivot[("Z2", "FULL")] / pivot[("Z2", "No-EQ")] - 1) * 100
if ("Z3", "FULL") in pivot.columns and ("Z3", "No-EQ") in pivot.columns:
    pivot["Gini_improve_pct"] = (1 - pivot[("Z3", "FULL")] / pivot[("Z3", "No-EQ")]) * 100

candidates = pivot.sort_values("Gini_improve_pct", ascending=False) if "Gini_improve_pct" in pivot.columns else pivot
candidates.head(10)


Z2                 Z3         Distance_gap_pct Gini_improve_pct
Variant        FULL      No-EQ    FULL   No-EQ                                  
Instance                                                                        
r202_21   1068.7063  1064.5402  0.0083  0.1044         0.391352        92.049808
rc203_21  1085.4009  1087.1797  0.0054  0.0603        -0.163616        91.044776
r203_21    899.4828   903.3312  0.0109  0.0828        -0.426023        86.835749
r211_21    767.2560   780.8277  0.0146  0.0595        -1.738117        75.462185
c205C10    228.2812   228.2812  0.0350  0.1253         0.000000        72.067039
r208_21    739.1960   737.5285  0.0051  0.0143         0.226093        64.335664
r209_21    874.9097   877.8584  0.0262  0.0677        -0.335897        61.299852
r207_21    821.3464   816.5645  0.0859  0.1709         0.585612        49.736688
rc205_21  1181.6437  1177.1917  0.0499  0.0894         0.378188        44.183445
c103C15    374.4278   374.4368  0.0668  0.1115        -0.002404        40.089686

## 5.3. Chọn case study #1: Gini cải thiện mạnh nhất với distance gap nhỏ

In [5]:
if "Distance_gap_pct" in candidates.columns and "Gini_improve_pct" in candidates.columns:
    case1_candidates = candidates[candidates["Distance_gap_pct"] < 5].sort_values(
        "Gini_improve_pct", ascending=False
    )
    print(case1_candidates.head(5))
    case1_instance = case1_candidates.index[0] if len(case1_candidates) else None
    print(f"\nCase study #1 đề xuất: {case1_instance}")


                 Z2                 Z3         Distance_gap_pct Gini_improve_pct
Variant        FULL      No-EQ    FULL   No-EQ                                  
Instance                                                                        
r202_21   1068.7063  1064.5402  0.0083  0.1044         0.391352        92.049808
rc203_21  1085.4009  1087.1797  0.0054  0.0603        -0.163616        91.044776
r203_21    899.4828   903.3312  0.0109  0.0828        -0.426023        86.835749
r211_21    767.2560   780.8277  0.0146  0.0595        -1.738117        75.462185
c205C10    228.2812   228.2812  0.0350  0.1253         0.000000        72.067039

Case study #1 đề xuất: r202_21


## 5.4. Trực quan hoá evolution trace cho case study đã chọn

In [6]:
import matplotlib.pyplot as plt

def plot_evolution_case(instance, variant, manifest):
    run = manifest[(manifest["Instance"] == instance) & (manifest["Variant"] == variant)].iloc[0]
    path = os.path.join(run["RunDir"], f"{run['Instance']}_seed_{int(run['Seed'])}_evolution_log.txt")
    ev = utils.parse_evolution_log(path)
    if ev.empty:
        print(f"Không parse được evolution log cho {instance} ({variant})")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ev["Iteration"], ev["Distance"], marker=".", alpha=0.6)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Distance (Z2) tại thời điểm accept/dominate")
    ax.set_title(f"Evolution trace — {instance} ({variant})")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{ARTIFACT_DIR}/fig_evolution_{instance}_{variant}.png", dpi=150)
    plt.show()

# Ví dụ gọi (chỉnh case1_instance theo kết quả cell 5.3):
# plot_evolution_case(case1_instance, "FULL", manifest)
# plot_evolution_case(case1_instance, "No-EQ", manifest)
print("Gọi plot_evolution_case(instance, variant, manifest) cho từng case study đã chọn ở 5.3.")


Gọi plot_evolution_case(instance, variant, manifest) cho từng case study đã chọn ở 5.3.


## 5.5. Bar chart active workload theo xe cho case study (dùng dữ liệu notebook 04)

In [7]:
import importlib
import evrp_analysis_utils as utils
importlib.reload(utils)


workload_path = f"{ARTIFACT_DIR}/route_workload_raw.csv"
if os.path.exists(workload_path):
    workload_all = pd.read_csv(workload_path)

    def plot_workload_bars(instance, workload_all):
        sub = workload_all[workload_all["Instance"] == instance]
        if sub.empty:
            print(f"Không có dữ liệu workload cho {instance}")
            return
        fig, ax = plt.subplots(figsize=(8, 4))
        for variant, g in sub.groupby("Variant"):
            ax.bar(
                [f"{variant}-V{i}" for i in range(len(g))],
                g["ActiveWorkload_A"], alpha=0.7, label=variant,
            )
        ax.set_ylabel("Active Workload A_r")
        ax.set_title(f"Active workload theo xe — {instance}")
        ax.legend()
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.savefig(f"{ARTIFACT_DIR}/fig_workload_bars_{instance}.png", dpi=150)
        plt.show()

    print("Gọi plot_workload_bars(instance, workload_all) cho case study đã chọn.")
else:
    print("Chạy notebook 04 trước để có route_workload_raw.csv")


Gọi plot_workload_bars(instance, workload_all) cho case study đã chọn.
